# AG_PRAXIS NB07b — Capture-Invariant Training

Every attack class in this dataset was recorded in its own session, and NB03 showed that a
model reading these columns can tell one recording from another: with the attack class held
fixed, so that telling attacks apart is no help at all, a forest names which recording a row
came from at 0.8280 against a chance rate of 0.1750.

The obvious response is to find the column responsible and drop it. NB03b tested that. It
took Duration out, which is the TTL header field and the one column in the timing family
that is not a measure of time, and reran NB03 unchanged. Identification with the attack
class held fixed did not fall. It rose, to 0.8504 from 0.8280. The timing family without
Duration still names the recording at 0.8935 against a chance rate of 0.0200.

So the session is not carried by one header field, and there is no column to drop. What
carries it is the same measured timing behaviour that carries the attack, which means
feature selection cannot separate the two without throwing away what the model is supposed
to detect. That is why this notebook intervenes on the representation instead: it trains the
model so that its internal representation is worse at telling the recordings apart, and then
asks which attacks come back.

The parent is the sequence model from NB06 and the single change is the training objective.
Everything else — the windows, the split, the architecture, the seed, the ten epochs at batch
32 — is the parent's, and this notebook asserts that rather than assuming it.

Two things are reported. Per-class F1 across all nineteen classes, so the run reads on the
same axis as the five class-imbalance interventions NB07 measured. And capture
identifiability off the learned representation, which is what says whether the objective did
what it was meant to do.

The usual first cell: mount Drive so the artefacts survive the session, clone the repository
so `src/` can be imported, and record the commit the code came from.

In [ ]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Configuration comes from `config/base.yaml` and the parent's own `config.json`, so nothing
about the run is typed here twice. The parent is `sequence_cnn_lstm_19class` from NB06 and
this notebook reads its metrics as well, because every comparison below is against it.

In [ ]:
import gc
import json
import random
import time

import numpy as np
import pandas as pd

from baselines import mohammadi as mo
from src import captures as cap
from src import interventions as iv
from src import inventory as inv
from src import runs as rn
from src import sequence as sq

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
BATCH_SIZE = int(CFG["training"]["batch_size"])
EPOCHS = int(CFG["training"]["epochs"])
WINDOW = int(CFG["sequence"]["window"])
STRIDE = int(CFG["sequence"]["stride"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])

FAST = os.environ.get("FAST", "0") == "1"
OUT_DIR = ARTIFACTS / ("NB07b_fast" if FAST else "NB07b")
OUT_DIR.mkdir(parents=True, exist_ok=True)

if IN_COLAB and not ARTIFACTS.exists():
    raise FileNotFoundError(
        f"{ARTIFACTS} does not exist. Drive is not mounted, or the artefacts path in "
        "config/base.yaml is wrong. Nothing this notebook writes would survive."
    )


def first_existing(candidates, what):
    found = next((Path(p) for p in candidates if Path(p).exists()), None)
    if found is None:
        raise FileNotFoundError(f"{what} not found. Looked in: {[str(p) for p in candidates]}")
    return found


MANIFEST_PATH = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB04_manifest.json",
     ARTIFACTS / "NB04" / "NB04_manifest.json"],
    "NB04_manifest.json",
)
PARENT_DIR = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB06",
     ARTIFACTS / "NB06" / "sequence_cnn_lstm_19class"],
    "the parent run's config.json and metrics.json",
)
ARRAY_DIR = first_existing([ARTIFACTS / "NB04"], "the NB04 sequence arrays")

MANIFEST = json.loads(MANIFEST_PATH.read_text())
PARENT = json.loads((PARENT_DIR / "config.json").read_text())
PARENT_METRICS = json.loads((PARENT_DIR / "metrics.json").read_text())

FEATURES = list(MANIFEST["columns"]["kept"])
CLASSES = sorted(MANIFEST["arrays"]["sequences_train"]["by_class"])
SEQUENCES = {
    part: dict(MANIFEST["arrays"][f"sequences_{part}"]["by_class"])
    for part in ("train", "val", "test")
}

pd.set_option("display.max_rows", 400)
pd.set_option("display.width", 240)

print(f"pass            : {'FAST, not a result' if FAST else 'FULL, the one the repository takes'}")
print(f"seed            : {SEED}")
print(f"window, stride  : {WINDOW}, {STRIDE}")
print(f"batch, epochs   : {BATCH_SIZE}, {EPOCHS}")
print(f"features        : {len(FEATURES)}")
print(f"classes         : {len(CLASSES)}")
print(f"parent          : {PARENT['run_id']}, macro F1 {PARENT_METRICS['macro_f1']:.4f}, "
      f"{PARENT_METRICS['n_parameters']:,} parameters")
print(f"arrays          : {ARRAY_DIR}")
print(f"artefacts       : {OUT_DIR}")

assert len(FEATURES) == 44, f"expected 44 features, got {len(FEATURES)}"
assert len(CLASSES) == 19, f"expected 19 classes, got {len(CLASSES)}"
assert (WINDOW, STRIDE) == (50, 25), f"window and stride are {WINDOW}, {STRIDE}"
assert (BATCH_SIZE, EPOCHS) == (32, 10), f"batch and epochs are {BATCH_SIZE}, {EPOCHS}"
assert MANIFEST["split"]["protocol"] == "two_tier", "the manifest is not the two-tier split"
assert int(PARENT_METRICS["n_parameters"]) == 214227, (
    f"the parent reports {PARENT_METRICS['n_parameters']:,} parameters, not 214,227"
)
assert int(PARENT_METRICS["n_test"]) == 49159, (
    f"the parent was scored on {PARENT_METRICS['n_test']:,} items, not 49,159"
)

The class sets are the ones already fixed. The five come from `PREREGISTRATION.md`
Amendment 6 and the F1 0.50 threshold from Amendment 5. The eight volumetric classes are the
group Section 3 of `PROJECT_RECORD.md` fixes from the benchmark taxonomy, and their mean is
reported because NB07 reported it for the five interventions and this run has to be readable
beside them.

The five NB07 runs are read from their own artefacts rather than copied in as numbers.

In [ ]:
DETECTED_AT = 0.50

FIVE = [
    "Recon-VulScan",
    "Recon-OS_Scan",
    "MQTT-DDoS-Publish_Flood",
    "Spoofing",
    "MQTT-Malformed_Data",
]
VOLUMETRIC = [
    "DDoS-ICMP", "DDoS-SYN", "DDoS-TCP", "DDoS-UDP",
    "DoS-ICMP", "DoS-SYN", "DoS-TCP", "DoS-UDP",
]
NB07_RUNS = [
    "class_weighted_loss", "focal_loss", "logit_adjustment",
    "threshold_tuning", "window_resampling",
]

NB07_DIR = first_existing(
    [REPO_ROOT / "results" / "NB07", ARTIFACTS / "NB07"], "the NB07 run directories"
)
NB07 = {}
for name in NB07_RUNS:
    path = NB07_DIR / name / "metrics.json"
    if path.exists():
        NB07[name] = json.loads(path.read_text())

def row_for(name, metrics):
    return {
        "run": name,
        **{c: metrics["per_class_f1"][c] for c in FIVE},
        "volumetric mean": float(np.mean([metrics["per_class_f1"][c] for c in VOLUMETRIC])),
        "macro F1": metrics["macro_f1"],
    }

STANDING = pd.DataFrame(
    [row_for(f"{PARENT['run_id']} (parent)", PARENT_METRICS)]
    + [row_for(name, NB07[name]) for name in NB07_RUNS if name in NB07]
)

print(f"NB07 runs read from {NB07_DIR}: {len(NB07)} of {len(NB07_RUNS)}")
print()
print("where this run has to be read, one row per run already measured")
print(STANDING.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print(f"of the five, the parent detects at F1 {DETECTED_AT:.2f}: "
      f"{sum(PARENT_METRICS['per_class_f1'][c] >= DETECTED_AT for c in FIVE)} of {len(FIVE)}")

assert len(NB07) == len(NB07_RUNS), (
    f"only {len(NB07)} of the five NB07 runs were found under {NB07_DIR}"
)

Seeding, and a look at what the session is running on. The seed is set before any model is
built.

In [ ]:
import keras
import tensorflow as tf

random.seed(SEED)
np.random.seed(SEED)
keras.utils.set_random_seed(SEED)

GPUS = tf.config.list_physical_devices("GPU")


def accelerator():
    """What this ran on, by name, so a wall time can be read against the hardware."""
    devices = tf.config.list_physical_devices("GPU")
    if not devices:
        return "cpu"
    named = []
    for device in devices:
        details = tf.config.experimental.get_device_details(device)
        named.append(str(details.get("device_name", device.name)))
    return ", ".join(named)


# Recorded into every run's config, so a run says what it ran under rather than
# leaving it to be remembered, and so a later dry run has something real to compare
# its own versions against.
ENVIRONMENT = {
    "tensorflow": tf.__version__,
    "keras": keras.__version__,
    "backend": keras.backend.backend(),
    "accelerator": accelerator(),
    "run_date": RUN_DATE,
}

print(f"seeded with : {SEED}")
print(f"tensorflow  : {tf.__version__}")
print(f"keras       : {keras.__version__}, backend {keras.backend.backend()}")
print(f"gpu         : {[d.name for d in GPUS] if GPUS else 'none, this will be slow'}")
print(f"environment : {ENVIRONMENT}")

assert keras.backend.backend() == "tensorflow", (
    "the group-weighted training loop takes its own gradients and has only been run on the "
    f"tensorflow backend; this session reports {keras.backend.backend()!r}"
)

The model is the parent's, unchanged. Building one here and checking it against the published
encoder is how the notebook establishes that, rather than taking it on trust: the encoder has
to match the baseline layer for layer and the parameter count has to be the parent's 214,227.

In [ ]:
SPECIMEN = sq.build_model(len(FEATURES), len(CLASSES), window=WINDOW, lstm_units=sq.LSTM_UNITS)
ENCODER_CHECK = sq.encoder_matches_baseline(SPECIMEN, len(FEATURES), len(CLASSES))

print("the record encoder against the published network, layer by layer")
print(pd.DataFrame(ENCODER_CHECK["rows"]).to_string(index=False))
print()
print(f"parameters : {SPECIMEN.count_params():,}")
print(f"input      : {SPECIMEN.input_shape}, output {SPECIMEN.output_shape}")
print(f"compiled   : {mo.COMPILE}")

assert ENCODER_CHECK["agrees"], "the encoder here is not the published encoder"
assert SPECIMEN.input_shape == (None, WINDOW, len(FEATURES), 1)
assert int(SPECIMEN.count_params()) == int(PARENT_METRICS["n_parameters"]), (
    "this model does not have the parent's parameter count, so it is not the same architecture"
)

del SPECIMEN
gc.collect()

Now the windows. The arrays are the ones NB04 wrote, read straight off Drive, and the checks
on them are the same ones NB07 ran: the columns and classes have to be the ones the manifest
lists, the window and stride have to be the ones the sequences were cut at, and the test
partition has to be the 49,159 windows every run in this project is scored on.

The recording each window came from is read alongside, because that is what the groups are
built from in the next cell.

In [ ]:
def read_partition(name):
    path = ARRAY_DIR / f"sequences_{name}.npz"
    if not path.exists():
        raise FileNotFoundError(f"{path} is missing. NB04 writes it.")
    with np.load(path, allow_pickle=False) as npz:
        if [str(v) for v in npz["features"]] != FEATURES:
            raise ValueError(f"{path.name} holds different columns than the manifest lists")
        if [str(v) for v in npz["classes"]] != CLASSES:
            raise ValueError(f"{path.name} holds different classes than the manifest lists")
        if (int(npz["window"]), int(npz["stride"])) != (WINDOW, STRIDE):
            raise ValueError(f"{path.name} was cut at a different window or stride")
        X = npz["X"]
        y = npz["y"].astype("int64")
        recording_codes = npz["recording"].astype("int64")
        recordings = [str(v) for v in npz["recordings"]]
    print(f"  {path.name:<24} {str(X.shape):>22}   {X.dtype}")
    assert X.shape[1:] == (WINDOW, len(FEATURES))
    assert np.isfinite(X).all(), f"{path.name} holds a value that is not finite"
    return {"X": X, "y": y, "recording_codes": recording_codes, "recordings": recordings}


print("reading the sequence arrays")
TRAIN = read_partition("train")
TEST = read_partition("test")

X_TRAIN = sq.reshape(TRAIN["X"])
X_TEST = sq.reshape(TEST["X"])

print()
print(f"train windows : {len(TRAIN['y']):,}")
print(f"test windows  : {len(TEST['y']):,}")
print(f"model input   : {X_TRAIN.shape[1:]}")

assert len(TEST["y"]) == 49159, f"the test partition holds {len(TEST['y']):,} windows, not 49,159"
assert len(TRAIN["y"]) == int(MANIFEST["arrays"]["sequences_train"]["shape"][0])

The groups. `PREREGISTRATION.md` Amendment 14 names the environment variable as `capture_id`
from the NB04 manifest. The manifest does not store a column of that name: what it stores per
window is the recording the window was cut from, which is the file stem, so `Benign_train`
and `Benign_test` are two recordings of one capture. The capture identifier is derived from
the recording name by `src/captures.py parse_capture`, which is the same rule NB01 and NB03
use, and that derivation is done here rather than assumed.

Two counts are worth printing rather than leaving implicit. The corpus holds 57 captures;
Amendment 14 records that figure. The objective is computed over training windows, and the
training partition holds fewer, because twelve captures appear only in validation or test and
the objective never sees them. Amendment 16 records the count the objective actually forms.

In [ ]:
def capture_of(recording_name):
    """The capture a recording belongs to, by the project's own naming rule."""
    return cap.parse_capture(f"{recording_name}.pcap.csv")["capture_id"]


CAPTURE_OF_RECORDING = {name: capture_of(name) for name in TRAIN["recordings"]}
TRAIN_CAPTURES = [CAPTURE_OF_RECORDING[TRAIN["recordings"][c]] for c in TRAIN["recording_codes"]]

GROUP_CODES, GROUP_NAMES = iv.group_codes(TRAIN_CAPTURES)
GROUP_SIZES = iv.group_sizes(GROUP_CODES, len(GROUP_NAMES))
N_GROUPS = len(GROUP_NAMES)

CORPUS_CAPTURES = sorted({capture_of(b["recording"]) for b in MANIFEST["split"]["blocks"]})
UNSEEN = sorted(set(CORPUS_CAPTURES) - set(GROUP_NAMES))

CLASS_OF_GROUP = {}
for code, label_code in zip(GROUP_CODES, TRAIN["y"]):
    CLASS_OF_GROUP.setdefault(GROUP_NAMES[code], CLASSES[label_code])
CAPTURES_PER_CLASS = {}
for group, label in CLASS_OF_GROUP.items():
    CAPTURES_PER_CLASS.setdefault(label, []).append(group)
MULTI = {k: v for k, v in CAPTURES_PER_CLASS.items() if len(v) > 1}
SINGLE = {k: v for k, v in CAPTURES_PER_CLASS.items() if len(v) == 1}

print(f"recordings in the training partition : {len(set(TRAIN['recording_codes'].tolist()))}")
print(f"captures the objective forms groups from : {N_GROUPS}")
print(f"captures in the corpus                   : {len(CORPUS_CAPTURES)}")
print(f"captures the objective never sees        : {len(UNSEEN)}   {', '.join(UNSEEN)}")
print()
print(f"classes with more than one training capture : {len(MULTI)}   "
      f"{ {k: len(v) for k, v in sorted(MULTI.items())} }")
print(f"classes with exactly one                    : {len(SINGLE)}")
print()
print("For the single-capture classes the group is the class, so on those the objective and "
      "a class")
print("weighting are the same operation. All five of the classes fixed in Amendment 6 are "
      "among them:")
for c in FIVE:
    print(f"  {c:<26} {len(CAPTURES_PER_CLASS[c])} capture, "
          f"{int(GROUP_SIZES[GROUP_NAMES.index(CAPTURES_PER_CLASS[c][0])]):,} training windows")
print()
sizes = pd.DataFrame({
    "capture": GROUP_NAMES,
    "class": [CLASS_OF_GROUP[g] for g in GROUP_NAMES],
    "train_windows": GROUP_SIZES,
}).sort_values("train_windows")
print("the five smallest groups and the three largest")
print(pd.concat([sizes.head(5), sizes.tail(3)]).to_string(index=False))
print()
print(f"largest to smallest: {sizes['train_windows'].max() / sizes['train_windows'].min():,.0f} to 1")

assert N_GROUPS == 45, f"the training partition forms {N_GROUPS} groups, not 45"
assert len(CORPUS_CAPTURES) == 57, f"the corpus holds {len(CORPUS_CAPTURES)} captures, not 57"
assert len(UNSEEN) == 12, f"{len(UNSEEN)} captures are unseen, not 12"
assert len(MULTI) == 8 and len(SINGLE) == 11
assert all(len(CAPTURES_PER_CLASS[c]) == 1 for c in FIVE)
assert int(GROUP_SIZES.sum()) == len(TRAIN["y"])

Before the intervention is read, the training loop it runs in has to be checked.

The objective needs a loop that knows which capture each window came from, which `model.fit`
cannot express, so `src/sequence.py` has one written out. A loop written out is a loop that
can be wrong, and if it is wrong then every figure below carries the error rather than the
intervention.

The check is to run that loop with the weighting turned into the parent's own objective and
see whether it lands where the parent landed. `GroupWeights` takes `batch_share`, which reads
its weights off the batch in front of it and makes the weighted loss exactly the mean loss
over the batch, with `eta` at zero so nothing moves. That is the parent's objective exactly,
so a run of it should score what the parent scored.

Exactly is not the test, though. The loop shuffles with its own seeded generator, so even at
seed 42 it draws different batches than `model.fit` did, and two ten-epoch runs on a GPU
differ anyway. The interval was fixed in `DECISIONS.md` before this ran: 0.713792, the
parent's own macro-F1, plus or minus 0.023312, the cross-seed spread NB08 measured for this
configuration over seeds 42 to 46. That is [0.690480, 0.737104]. It is a weak check and the
decision entry says so: it catches an error large enough to move the result outside the
spread of runs of the same configuration, and nothing finer.

In [ ]:
CHECK_CENTRE = 0.713792
CHECK_HALF_WIDTH = 0.023312
CHECK_INTERVAL = (CHECK_CENTRE - CHECK_HALF_WIDTH, CHECK_CENTRE + CHECK_HALF_WIDTH)
ETA = 1e-4

print(f"parent macro F1 on record : {PARENT_METRICS['macro_f1']:.6f}")
print(f"check centre              : {CHECK_CENTRE:.6f}")
print(f"check half-width          : {CHECK_HALF_WIDTH:.6f}, the NB08 cross-seed spread")
print(f"check interval            : [{CHECK_INTERVAL[0]:.6f}, {CHECK_INTERVAL[1]:.6f}]")
print(f"eta for the run below     : {ETA:g}, fixed in DECISIONS.md and not tuned")

assert abs(PARENT_METRICS["macro_f1"] - CHECK_CENTRE) < 1e-6, (
    "the interval is centred on a figure that is not the parent's macro-F1 as read from its "
    "own metrics.json"
)

The check run. `eta` is zero and the weights come from each batch, so this is the parent's
objective through the new loop. It writes its own run directory like any other run, because a
check that leaves nothing behind cannot be looked at again.

In [ ]:
CHECK_CONFIG = dict(PARENT)
CHECK_CONFIG["run_id"] = "loop_check_batch_share"
CHECK_CONFIG["parent"] = PARENT["run_id"]
CHECK_CONFIG["training_objective"] = "batch_share, eta 0, which is the mean loss over the batch"
CHECK_CONFIG["observed"] = dict(PARENT.get("observed", {}))
CHECK_CONFIG["observed"].update({
    "environment": ENVIRONMENT,
    "run_date": RUN_DATE,
    "mode": "fast" if FAST else "full",
    "purpose": "check that the group-weighted loop reproduces the parent, before the "
               "intervention is read",
    "check_interval": list(CHECK_INTERVAL),
    "n_groups": N_GROUPS,
})

CHECK_WEIGHTS = iv.worst_group_dro(GROUP_SIZES, eta=0.0, start="batch_share")

CHECK_RUN = sq.fit_and_save(
    OUT_DIR,
    CHECK_CONFIG["run_id"],
    X_train=X_TRAIN,
    y_train=TRAIN["y"],
    X_test=X_TEST,
    y_test=TEST["y"],
    classes=CLASSES,
    config=CHECK_CONFIG,
    parent=PARENT,
    window=WINDOW,
    n_features=len(FEATURES),
    lstm_units=sq.LSTM_UNITS,
    seed=SEED,
    checkpoint=True,
    verbose=2,
    groups=GROUP_CODES,
    group_weights=CHECK_WEIGHTS,
    group_names=GROUP_NAMES,
)

In [ ]:
CHECK_MACRO = float(CHECK_RUN["metrics"]["macro_f1"])
CHECK_PASSES = bool(CHECK_INTERVAL[0] <= CHECK_MACRO <= CHECK_INTERVAL[1])

print("#" * 100)
if CHECK_PASSES:
    print("#  THE LOOP REPRODUCES THE PARENT")
else:
    print("#  THE LOOP DOES NOT REPRODUCE THE PARENT")
print("#" * 100)
print(f"  parent          : {CHECK_CENTRE:.6f}")
print(f"  loop, batch_share: {CHECK_MACRO:.6f}")
print(f"  difference      : {CHECK_MACRO - CHECK_CENTRE:+.6f}")
print(f"  interval        : [{CHECK_INTERVAL[0]:.6f}, {CHECK_INTERVAL[1]:.6f}]")
print()
if CHECK_PASSES:
    print("  The loop runs the parent's objective to within the spread of runs of this")
    print("  configuration, so what the intervention below changes is the objective and not")
    print("  the machinery. This is a weak check: it would not catch a small error.")
else:
    print("  The loop differs from model.fit in some way not yet found. Every figure below")
    print("  carries that difference, and nothing in this notebook should be reported until")
    print("  it is explained.")
print("#" * 100)

Now the intervention. The objective becomes worst-group: the loss the gradient is taken of is
a weighted mean of the captures' mean losses rather than the mean over the windows, and the
weights move by exponentiated gradient towards whichever captures the model is doing worst
on. `eta` is 1e-4, fixed before the run and not tuned, with the derivation in `DECISIONS.md`.

The weights start uniform, at one forty-fifth each. `PREREGISTRATION.md` Amendment 17 records
what that means: from the first update a capture of 24 windows and a capture of 8,290 carry
the same weight, so the run departs from its parent by group balancing before worst-group
weighting has moved anything, and on the five classes above the group is the class.

There is no group-size floor, no weight cap and no warmup. The weights are written into
`metrics.json` at the end of every epoch, so if the objective collapses onto the smallest
captures that is in the artefact.

In [ ]:
RUN_CONFIG = dict(PARENT)
RUN_CONFIG["run_id"] = "capture_invariant_dro"
RUN_CONFIG["parent"] = PARENT["run_id"]
RUN_CONFIG["training_objective"] = "worst_group_dro"
RUN_CONFIG["observed"] = dict(PARENT.get("observed", {}))
RUN_CONFIG["observed"].update({
    "environment": ENVIRONMENT,
    "run_date": RUN_DATE,
    "mode": "fast" if FAST else "full",
    "changed_from_parent": ["training_objective"],
    "group_variable": "capture_id, derived from the recording name by captures.parse_capture",
    "n_groups": N_GROUPS,
    "captures_in_corpus": len(CORPUS_CAPTURES),
    "captures_unseen_by_the_objective": UNSEEN,
    "eta": ETA,
    "weights_start": "uniform",
    "registered_under": "PREREGISTRATION.md Amendments 14, 16 and 17",
    "loop_check": {
        "run": CHECK_CONFIG["run_id"],
        "macro_f1": CHECK_MACRO,
        "interval": list(CHECK_INTERVAL),
        "passes": CHECK_PASSES,
    },
})

print("one change from the parent:",
      sorted(rn.assert_single_change(RUN_CONFIG, PARENT)))
print(f"loss unchanged at {RUN_CONFIG['loss']!r}; the per-sample loss is still cross-entropy")
print()

WEIGHTS = iv.worst_group_dro(GROUP_SIZES, eta=ETA, start="uniform")

RUN = sq.fit_and_save(
    OUT_DIR,
    RUN_CONFIG["run_id"],
    X_train=X_TRAIN,
    y_train=TRAIN["y"],
    X_test=X_TEST,
    y_test=TEST["y"],
    classes=CLASSES,
    config=RUN_CONFIG,
    parent=PARENT,
    window=WINDOW,
    n_features=len(FEATURES),
    lstm_units=sq.LSTM_UNITS,
    seed=SEED,
    checkpoint=True,
    verbose=2,
    groups=GROUP_CODES,
    group_weights=WEIGHTS,
    group_names=GROUP_NAMES,
)

The first dependent variable: per-class F1 across all nineteen classes, the five, the
volumetric mean and macro-F1, each against the parent and against the five interventions NB07
measured. All of these runs are scored on the same 49,159 test windows, so every difference
is a difference on the same items.

In [ ]:
M = RUN["metrics"]

FULL_TABLE = pd.DataFrame({
    "class": CLASSES,
    "parent": [PARENT_METRICS["per_class_f1"][c] for c in CLASSES],
    "capture_invariant_dro": [M["per_class_f1"][c] for c in CLASSES],
    "test_windows": [SEQUENCES["test"][c] for c in CLASSES],
})
FULL_TABLE["difference"] = FULL_TABLE["capture_invariant_dro"] - FULL_TABLE["parent"]
print("all nineteen classes, against the parent")
print(FULL_TABLE.sort_values("difference").to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()

BESIDE = pd.concat([STANDING, pd.DataFrame([row_for("capture_invariant_dro", M)])])
print("this run beside the parent and the five NB07 interventions, same 49,159 windows")
print(BESIDE.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()

detected = sum(M["per_class_f1"][c] >= DETECTED_AT for c in FIVE)
parent_detected = sum(PARENT_METRICS["per_class_f1"][c] >= DETECTED_AT for c in FIVE)
volumetric_mean = float(np.mean([M["per_class_f1"][c] for c in VOLUMETRIC]))
parent_volumetric = float(np.mean([PARENT_METRICS["per_class_f1"][c] for c in VOLUMETRIC]))

print(f"macro F1            : {M['macro_f1']:.4f} against the parent's "
      f"{PARENT_METRICS['macro_f1']:.4f}, {M['macro_f1'] - PARENT_METRICS['macro_f1']:+.4f}")
print(f"volumetric mean     : {volumetric_mean:.4f} against {parent_volumetric:.4f}, "
      f"{volumetric_mean - parent_volumetric:+.4f}")
print(f"of the five at {DETECTED_AT:.2f}  : {detected} against {parent_detected} for the parent")
print(f"accuracy            : {M['accuracy']:.4f}, weighted F1 {M['weighted_f1']:.4f}")
print(f"trained in          : {M['train_seconds']:,.0f}s")
print()
print("No significance test is computed here. That is NB08's.")

What the weights did. This is where a collapse onto the smallest captures would show, and it
is recorded rather than prevented.

In [ ]:
DRO = M["group_dro"]
traj = pd.DataFrame([
    {
        "epoch": e["epoch"],
        "max weight": e["max_weight"],
        "min weight": e["min_weight"],
        "effective groups": e["effective_groups"],
        "heaviest capture": e["heaviest"][0]["group"],
        "its windows": e["heaviest"][0]["windows"],
    }
    for e in DRO["trajectory"]
])
print(f"start {DRO['start']}, eta {DRO['eta']:g}, {DRO['n_groups']} groups, "
      f"uniform weight would be {1 / DRO['n_groups']:.4f}")
print()
print(traj.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
last = DRO["trajectory"][-1]
print("the five heaviest captures at the end")
for entry in last["heaviest"]:
    print(f"  {entry['group']:<28} weight {entry['weight']:.4f}   "
          f"{entry['windows']:,} training windows   class {CLASS_OF_GROUP[entry['group']]}")
print()
appeared = pd.Series(DRO["batches_each_group_appeared_in"])
print(f"batches each group appeared in: smallest {appeared.min():,}, largest {appeared.max():,}")

The second dependent variable, and the arithmetic it needs before it can be run.

Amendment 14 asks for capture identifiability measured off the learned representation, using
the protocol NB03 used. NB03's protocol was defined on records: a forest naming which of 50
recordings a row came from, chance 1/50 = 0.0200, and the same with the attack class held
fixed across the 8 classes recorded more than once, mean chance 0.1750. Neither number
carries over to windows unchanged, and Amendment 16 fixes what replaces them.

Two things differ. The unit is a window rather than a record, and the partition is the
training one rather than test. The partition changed because the test and validation
partitions hold exactly one capture per class, so with the class held fixed there is nothing
to tell apart, and pooled across captures the capture and the class are the same label. Only
the training partition reproduces the structure NB03 measured on.

So the chance rates are recomputed here rather than reused. The probe runs on the 8 classes
that have more than one training capture, which hold 34 captures between them. Pooled, chance
is one over 34. With the class held fixed, chance is the mean of one over the number of
captures that class has. NB03's equal-draw rule carries over: every capture contributes the
same number of windows, so no capture can be identified by being larger than the others.

In [ ]:
PROBE_CLASSES = sorted(MULTI)
PROBE_CAPTURES = sorted({g for c in PROBE_CLASSES for g in CAPTURES_PER_CLASS[c]})
POOLED_CHANCE = 1.0 / len(PROBE_CAPTURES)
HELD_FIXED_CHANCE = float(np.mean([1.0 / len(CAPTURES_PER_CLASS[c]) for c in PROBE_CLASSES]))

probe_sizes = {g: int(GROUP_SIZES[GROUP_NAMES.index(g)]) for g in PROBE_CAPTURES}
DRAW = min(probe_sizes.values())

print(f"classes with more than one training capture : {len(PROBE_CLASSES)}   "
      f"{', '.join(PROBE_CLASSES)}")
print(f"captures across them                        : {len(PROBE_CAPTURES)}")
print()
print(f"pooled chance          : 1/{len(PROBE_CAPTURES)} = {POOLED_CHANCE:.4f}   "
      f"(NB03 on records: 1/50 = 0.0200)")
print(f"class-held-fixed chance: mean of 1/n over the {len(PROBE_CLASSES)} classes = "
      f"{HELD_FIXED_CHANCE:.4f}   (NB03 on records: 0.1750)")
print()
print("They differ from NB03's because NB03 counted a capture's train file and its test file")
print("as two recordings and reached 50, while the unit here is the capture and only its")
print("training side exists.")
print()
print(f"windows per capture : smallest {min(probe_sizes.values()):,}, "
      f"largest {max(probe_sizes.values()):,}")
print(f"equal draw          : {DRAW:,} windows from every capture, "
      f"{DRAW * len(PROBE_CAPTURES):,} in total")

assert len(PROBE_CAPTURES) == 34, f"{len(PROBE_CAPTURES)} captures, not 34"
assert abs(POOLED_CHANCE - 0.0294) < 5e-4
assert abs(HELD_FIXED_CHANCE - 0.2812) < 5e-4

The probe itself. The representation is the LSTM output for a window, which is the 128 numbers
the model has left after reading all fifty records and before the softmax turns them into a
class. A forest is fitted on those numbers to name the capture, with NB03's settings read out
of the file NB03 wrote rather than typed here.

The same probe is run on the parent's representation as well. Amendment 14 asks only for this
run's figure against NB03's, but NB03's figures were measured on raw records with a different
forest and a different unit, so they cannot say whether the objective changed anything. The
parent's representation can, and it costs one model load and one forward pass.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

VERDICT_PATH = first_existing(
    [REPO_ROOT / "data" / "processed" / "NB03_verdict.json",
     ARTIFACTS / "NB03" / "NB03_verdict.json"], "NB03_verdict.json"
)
NB03 = json.loads(VERDICT_PATH.read_text())
NB03_FOREST = NB03["forest"]

print(f"NB03 read from {VERDICT_PATH}")
print(f"  forest              : {NB03_FOREST}")
print(f"  all 44 features     : {NB03['capture_identification'][0]['accuracy']:.4f}")
print(f"  attack class fixed  : {NB03['within_class_mean_accuracy']:.4f}")

assert NB03_FOREST["n_estimators"] == 50 and NB03_FOREST["min_samples_leaf"] == 100
assert NB03_FOREST["test_fraction"] == 0.30 and NB03_FOREST["random_state"] == SEED


def representation(model, X, index, chunk=8192, batch=512):
    """The LSTM output for each window: 128 numbers, before the softmax.

    The windows are read a chunk of the index at a time rather than by slicing the
    whole index out of X first. That slice would be a second copy of a gigabyte of
    windows, and whether it fits would depend on which runtime the session got
    rather than on anything about the experiment.
    """
    reader = keras.Model(model.inputs, model.get_layer("across_the_window").output)
    pieces = []
    for start in range(0, len(index), int(chunk)):
        taken = index[start : start + int(chunk)]
        pieces.append(np.asarray(reader.predict(X[taken], batch_size=int(batch), verbose=0)))
    return np.concatenate(pieces)


def equal_draw(seed):
    """An index into the training windows taking the same count from every capture."""
    rng = np.random.default_rng(int(seed))
    keep = []
    for group in PROBE_CAPTURES:
        where = np.flatnonzero(GROUP_CODES == GROUP_NAMES.index(group))
        keep.append(rng.choice(where, size=DRAW, replace=False))
    return np.sort(np.concatenate(keep))


def probe(H, index, name):
    """NB03's forest on the representation, pooled and with the class held fixed."""
    target = np.asarray([GROUP_NAMES[c] for c in GROUP_CODES[index]])
    labels = np.asarray([CLASSES[c] for c in TRAIN["y"][index]])
    rows = []

    def one(mask, scope, chance):
        Xa, ya = H[mask], target[mask]
        Xtr, Xte, ytr, yte = train_test_split(
            Xa, ya, test_size=NB03_FOREST["test_fraction"],
            random_state=SEED, stratify=ya
        )
        forest = RandomForestClassifier(
            n_estimators=NB03_FOREST["n_estimators"],
            min_samples_leaf=NB03_FOREST["min_samples_leaf"],
            n_jobs=-1, random_state=SEED,
        )
        forest.fit(Xtr, ytr)
        forest.n_jobs = 1
        accuracy = float((forest.predict(Xte) == yte).mean())
        rows.append({"model": name, "scope": scope, "captures": int(len(set(ya))),
                     "chance": float(chance), "accuracy": accuracy})

    one(np.ones(len(target), dtype=bool), "pooled", POOLED_CHANCE)
    for label in PROBE_CLASSES:
        one(labels == label, f"within {label}", 1.0 / len(CAPTURES_PER_CLASS[label]))
    return pd.DataFrame(rows)


INDEX = equal_draw(SEED)
print()
print(f"probing {len(INDEX):,} training windows, {DRAW:,} from each of "
      f"{len(PROBE_CAPTURES)} captures")

PARENT_MODEL_PATH = first_existing(
    [ARTIFACTS / "NB06" / "sequence_cnn_lstm_19class" / "model.keras",
     PARENT_DIR / "model.keras"], "the parent's saved model"
)
PROBES = []
for name, model_path in (("capture_invariant_dro", Path(RUN["model_file"])),
                         ("sequence_cnn_lstm_19class (parent)", PARENT_MODEL_PATH)):
    model = keras.models.load_model(model_path)
    H = representation(model, X_TRAIN, INDEX)
    print(f"  {name:<38} representation {H.shape}")
    PROBES.append(probe(H, INDEX, name))
    del model, H
    gc.collect()

PROBE_TABLE = pd.concat(PROBES, ignore_index=True)

In [ ]:
pooled = PROBE_TABLE[PROBE_TABLE["scope"] == "pooled"]
held = PROBE_TABLE[PROBE_TABLE["scope"] != "pooled"]
held_mean = held.groupby("model")["accuracy"].mean()

print("capture identification off the learned representation, training windows")
print(PROBE_TABLE.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print()
print(f"{'model':<40} {'pooled':>10} {'class fixed':>14}")
print("-" * 68)
for name in PROBE_TABLE["model"].unique():
    p = float(pooled[pooled['model'] == name]['accuracy'].iloc[0])
    print(f"{name:<40} {p:>10.4f} {held_mean[name]:>14.4f}")
print("-" * 68)
print(f"{'chance':<40} {POOLED_CHANCE:>10.4f} {HELD_FIXED_CHANCE:>14.4f}")
print()
print("NB03, for reference, on records rather than windows and with 50 recordings rather")
print(f"than 34 captures: {NB03['capture_identification'][0]['accuracy']:.4f} pooled at "
      f"chance 0.0200, and {NB03['within_class_mean_accuracy']:.4f} with the class held")
print("fixed at chance 0.1750. Different unit, different partition and a different number of")
print("targets, so the NB03 figures locate this measurement rather than pair with it.")

The ledger entry, ready to paste into `RESULTS_LEDGER.md`.

In [ ]:
if FAST:
    status = "DO NOT ENTER, fast pass"
elif not CHECK_PASSES:
    status = "DO NOT ENTER, the loop did not reproduce the parent"
elif GIT_DIRTY:
    status = "reported result, working tree dirty"
else:
    status = "reported result, an intervention under H1"

dro_last = DRO["trajectory"][-1]
ledger = f"""
### NB07b — capture-invariant training ({RUN_DATE})

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB07b_capture_invariant.ipynb |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| status | {status} |
| registered under | PREREGISTRATION.md Amendments 14, 16 and 17 |
| parent | {PARENT['run_id']}, one change: training_objective |
| objective | worst-group DRO, exponentiated gradient, eta {ETA:g}, weights start uniform, no floor or cap or warmup |
| group variable | capture_id, derived from the recording name by captures.parse_capture |
| groups | {N_GROUPS} over training windows; {len(CORPUS_CAPTURES)} captures in the corpus, {len(UNSEEN)} never seen by the objective |
| group is the class | 11 of {N_GROUPS} groups, including all five Amendment 6 classes |
| test items | {M['n_test']:,} windows, the same as the parent and the five NB07 runs |
| parameters | {M['n_parameters']:,} |
| loop check, batch_share at eta 0 | {CHECK_MACRO:.6f} against the parent's {CHECK_CENTRE:.6f}, interval [{CHECK_INTERVAL[0]:.6f}, {CHECK_INTERVAL[1]:.6f}], {"passes" if CHECK_PASSES else "FAILS"} |
| macro F1 | {M['macro_f1']:.4f} against the parent's {PARENT_METRICS['macro_f1']:.4f}, {M['macro_f1'] - PARENT_METRICS['macro_f1']:+.4f} |
| volumetric mean | {volumetric_mean:.4f} against {parent_volumetric:.4f}, {volumetric_mean - parent_volumetric:+.4f} |
| of the five at F1 0.50 | {detected} against {parent_detected} for the parent |
| the five | {" · ".join(f"{c} {M['per_class_f1'][c]:.4f} ({PARENT_METRICS['per_class_f1'][c]:.4f})" for c in FIVE)} |
| heaviest group weight at the end | {dro_last['max_weight']:.4f} on {dro_last['heaviest'][0]['group']}, {dro_last['heaviest'][0]['windows']:,} windows, against a uniform {1 / N_GROUPS:.4f} |
| effective groups at the end | {dro_last['effective_groups']:.2f} of {N_GROUPS} |
| capture identification, this run's representation | {float(pooled[pooled['model'] == 'capture_invariant_dro']['accuracy'].iloc[0]):.4f} pooled at chance {POOLED_CHANCE:.4f}, {held_mean['capture_invariant_dro']:.4f} with the class held fixed at chance {HELD_FIXED_CHANCE:.4f} |
| capture identification, the parent's representation | {float(pooled[pooled['model'] != 'capture_invariant_dro']['accuracy'].iloc[0]):.4f} pooled, {held_mean[[i for i in held_mean.index if i != 'capture_invariant_dro'][0]]:.4f} with the class held fixed |
| probe | {DRAW:,} training windows from each of {len(PROBE_CAPTURES)} captures across {len(PROBE_CLASSES)} classes, NB03's forest settings, on the 128-number LSTM output |
| seed replicates | none. Single run at seed {SEED} |
| not computed | significance. That is NB08's |
| artefacts | {OUT_DIR} |

Duration stays in the 44 features. The objective changes what the model optimises, not what it reads.
"""

print(ledger)